# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [12]:
from pyspark.sql.functions import (
    col, monotonically_increasing_id, avg, max as spark_max, min as spark_min,
    unix_timestamp, to_date, dayofweek, hour, count, corr, when
)

In [6]:
# Part 1
# Add a column that creates a unique key to identify each record in order to answer questions about individual trips



df_trips = df_trips.withColumn("trip_id", monotonically_increasing_id())


In [7]:
# uncomment to see the trip_id 
# these id are just random different non chronological numbers 


# df_trips.show()

In [8]:
# Which trip has the highest passanger count


df_trips.orderBy(col("passenger_count").desc()).limit(1).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       2| 2019-01-05 13:12:29|  2019-01-05 13:12:32|            9.0|          0.0|       5.0|                 N|          68|          68|           1

In [9]:
# What is the Average passanger count

df_trips.agg(avg("passenger_count")).show()

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [14]:
# Shortest/longest trip by distance? by time?.
df_trips = df_trips.withColumn(
    "trip_duration_sec",
    unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")
)

df_trips = df_trips.withColumn("pickup_date", to_date("tpep_pickup_datetime"))
df_trips = df_trips.withColumn("pickup_hour", hour("tpep_pickup_datetime"))
df_trips = df_trips.withColumn("pickup_dow", dayofweek("tpep_pickup_datetime"))  # 1=Sunday ... 7=Saturday


# longest, distance
df_trips.orderBy(col("trip_distance").desc()).limit(1).show()
 # shortest, distance
df_trips.orderBy(col("trip_distance").asc()).limit(1).show()   
 # longest, time
df_trips.orderBy(col("trip_duration_sec").desc()).limit(1).show() 
 # shortest, time
df_trips.orderBy(col("trip_duration_sec").asc()).limit(1).show()  

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+-----------------+-----------+-----------+----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|trip_duration_sec|pickup_date|pickup_hour|pickup_dow|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+-----------------+-----------+-----------+----

In [18]:
# busiest day/slowest single day
daily_counts = df_trips.groupBy("pickup_date").agg(count("*").alias("trip_count"))

# busiest day
daily_counts.orderBy(col("trip_count").desc()).limit(1).show()  
# slowest day
daily_counts.orderBy(col("trip_count").asc()).limit(1).show()   

+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-01-25|    292499|
+-----------+----------+

+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-02-23|         1|
+-----------+----------+



In [19]:
# busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )

df_trips = df_trips.withColumn(
    "time_bucket",
    when((col("pickup_hour") >= 5) & (col("pickup_hour") < 12), "Morning")
    .when((col("pickup_hour") >= 12) & (col("pickup_hour") < 17), "Afternoon")
    .when((col("pickup_hour") >= 17) & (col("pickup_hour") < 21), "Evening")
    .otherwise("Late Night")
)
df_trips.createOrReplaceTempView("trips")  # refresh view with new column

bucket_counts = df_trips.groupBy("time_bucket").agg(count("*").alias("trip_count"))
bucket_counts.orderBy(col("trip_count").desc()).show()

+-----------+----------+
|time_bucket|trip_count|
+-----------+----------+
|  Afternoon|   2111999|
|    Morning|   2035497|
|    Evening|   1882211|
| Late Night|   1666910|
+-----------+----------+



In [20]:
# On average which day of the week is slowest/busiest
# 1=Sunday, 2=Monday, ... 7=Saturday : Spark's dayofweek convention
dow_avg = (
    df_trips.groupBy("pickup_dow", "pickup_date")
    .agg(count("*").alias("daily_count"))
    .groupBy("pickup_dow")
    .agg(avg("daily_count").alias("avg_trips"))
    .orderBy(col("avg_trips").desc())
)
dow_avg.show()

+----------+------------------+
|pickup_dow|         avg_trips|
+----------+------------------+
|         5| 193863.2857142857|
|         6|155316.42857142858|
|         7|144283.57142857142|
|         4|140584.88888888888|
|         3|          120908.4|
|         1|        107488.125|
|         2|           90812.1|
+----------+------------------+



In [21]:
# Does trip distance or num passangers affect tip amount
df_trips.select(
    corr("trip_distance", "tip_amount").alias("corr_distance_tip"),
    corr("passenger_count", "tip_amount").alias("corr_passengers_tip")
).show()
# the trip distance explain upto 52% of the tip amount 

+------------------+--------------------+
| corr_distance_tip| corr_passengers_tip|
+------------------+--------------------+
|0.5269200663652668|0.001084223312167...|
+------------------+--------------------+



In [22]:
# What was the highest "extra" charge and which trip
df_trips.orderBy(col("extra").desc()).limit(1).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+-----------------+-----------+-----------+----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount| extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|trip_duration_sec|pickup_date|pickup_hour|pickup_dow|time_bucket|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+-----------------+-

In [23]:
# Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

df_trips.select("trip_distance", "trip_duration_sec", "passenger_count", "fare_amount", "tip_amount", "extra").describe().show()

+-------+------------------+-----------------+------------------+-----------------+------------------+------------------+
|summary|     trip_distance|trip_duration_sec|   passenger_count|      fare_amount|        tip_amount|             extra|
+-------+------------------+-----------------+------------------+-----------------+------------------+------------------+
|  count|           7696617|          7696617|           7667945|          7696617|           7696617|           7696617|
|   mean|2.8301461681153532|993.0648942256058|1.5670317144945614|12.52967677747685|1.8208300763883147|0.3374054146126797|
| stddev| 3.774548394256295|4900.523766730928|1.2244198591042095|261.5897471783846|2.4994631914320986|0.5313564053935059|
|    min|               0.0|         -5056830|               0.0|           -362.0|             -63.5|             -60.0|
|    max|             831.8|          2618881|               9.0|        623259.86|            787.25|            535.38|
+-------+---------------

**Outlier reasoning:** Looking at `.describe()`, I flagged the following as likely data errors rather than real trips:

- `trip_distance = 0` with `fare_amount > 0` : GPS/meter glitch, not a real trip
- `passenger_count = 0` or unusually high (e.g. > 6) : sensor misreads, since standard taxis cap around 4–6
- `trip_duration_sec <= 0` or extremely large (e.g. > 24 hrs) : meter left running, or dropoff/pickup timestamp logging error
- `fare_amount` or `tip_amount` negative : refunds/corrections miscoded as trips

These aren't necessarily "wrong" data pipeline bugs  they reflect real-world sensor and logging noise common in taxi datasets, so I filtered them out before computing trip-level statistics rather than deleting them from the raw dataset entirely.tirely.tirely.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [24]:
# set dl url for taxi zone lookup
zone_lookup_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv'

# get the data
response = requests.get(zone_lookup_url)

# check that response was good and save the data
zone_lookup_file = "taxi_zone_lookup.csv"
if response.status_code == 200:
    with open(zone_lookup_file, "wb") as f:
        f.write(response.content)

# load into Spark
df_zones = spark.read.csv(zone_lookup_file, header=True, inferSchema=True)
df_zones.show(5)
df_zones.printSchema()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)



In [25]:
from pyspark.sql.functions import col

# Join for pickup borough
df_trips = df_trips.join(
    df_zones.select(col("LocationID").alias("PULocationID"), col("Borough").alias("pickup_borough")),
    on="PULocationID",
    how="left"
)

# Join for dropoff borough
df_trips = df_trips.join(
    df_zones.select(col("LocationID").alias("DOLocationID"), col("Borough").alias("dropoff_borough")),
    on="DOLocationID",
    how="left"
)

df_trips.createOrReplaceTempView("trips")

In [26]:
# which borough had most pickups? dropoffs?

df_trips.groupBy("pickup_borough").count().orderBy(col("count").desc()).show()

df_trips.groupBy("dropoff_borough").count().orderBy(col("count").desc()).show()

+--------------+-------+
|pickup_borough|  count|
+--------------+-------+
|     Manhattan|6950965|
|        Queens| 471173|
|       Unknown| 159815|
|      Brooklyn|  91905|
|         Bronx|  18062|
|           N/A|   3890|
|           EWR|    446|
| Staten Island|    361|
+--------------+-------+

+---------------+-------+
|dropoff_borough|  count|
+---------------+-------+
|      Manhattan|6817355|
|         Queens| 340972|
|       Brooklyn| 301105|
|        Unknown| 149097|
|          Bronx|  58085|
|            N/A|  16904|
|            EWR|  10914|
|  Staten Island|   2185|
+---------------+-------+



In [27]:
# what are the busy/slow times by borough

borough_hour = (
    df_trips.groupBy("pickup_borough", "pickup_hour")
    .count()
    .orderBy("pickup_borough", col("count").desc())
)
borough_hour.show(50)

# just the peak hour per borough
from pyspark.sql import Window
from pyspark.sql.functions import row_number

w = Window.partitionBy("pickup_borough").orderBy(col("count").desc())
borough_hour.withColumn("rank", row_number().over(w)).filter(col("rank") == 1).show()

+--------------+-----------+-----+
|pickup_borough|pickup_hour|count|
+--------------+-----------+-----+
|         Bronx|          7| 1803|
|         Bronx|          8| 1445|
|         Bronx|          6| 1301|
|         Bronx|          9| 1158|
|         Bronx|         10| 1079|
|         Bronx|         14| 1014|
|         Bronx|         12|  956|
|         Bronx|         13|  918|
|         Bronx|         15|  897|
|         Bronx|         11|  885|
|         Bronx|         17|  812|
|         Bronx|         16|  756|
|         Bronx|         18|  741|
|         Bronx|          5|  736|
|         Bronx|         19|  519|
|         Bronx|         20|  440|
|         Bronx|         23|  410|
|         Bronx|         21|  408|
|         Bronx|          4|  403|
|         Bronx|         22|  353|
|         Bronx|          0|  324|
|         Bronx|          1|  254|
|         Bronx|          3|  225|
|         Bronx|          2|  225|
|      Brooklyn|          8| 6935|
|      Brooklyn|    

In [28]:
# what are the busiest days of the week by borough?
borough_dow = (
    df_trips.groupBy("pickup_borough", "pickup_dow")
    .count()
    .orderBy("pickup_borough", col("count").desc())
)

w = Window.partitionBy("pickup_borough").orderBy(col("count").desc())
borough_dow.withColumn("rank", row_number().over(w)).filter(col("rank") == 1).show()

+--------------+----------+-------+----+
|pickup_borough|pickup_dow|  count|rank|
+--------------+----------+-------+----+
|         Bronx|         5|   3121|   1|
|      Brooklyn|         3|  15779|   1|
|           EWR|         4|     83|   1|
|     Manhattan|         5|1229554|   1|
|           N/A|         3|    703|   1|
|        Queens|         5|  78972|   1|
| Staten Island|         6|     64|   1|
|       Unknown|         5|  28929|   1|
+--------------+----------+-------+----+



In [29]:
# what is the average trip distance by borough?
from pyspark.sql.functions import avg

df_trips.groupBy("pickup_borough").agg(avg("trip_distance").alias("avg_distance")).orderBy(col("avg_distance").desc()).show()

+--------------+------------------+
|pickup_borough|      avg_distance|
+--------------+------------------+
| Staten Island|12.503601108033246|
|        Queens|11.283218499361993|
|         Bronx| 7.233194552098303|
|      Brooklyn| 4.787677275447492|
|           N/A| 3.193850899742941|
|           EWR| 2.641098654708519|
|       Unknown| 2.415464130400774|
|     Manhattan|2.2286693358402596|
+--------------+------------------+



In [30]:
# what is the average trip fare by borough?
df_trips.groupBy("pickup_borough").agg(avg("fare_amount").alias("avg_fare")).orderBy(col("avg_fare").desc()).show()

+--------------+------------------+
|pickup_borough|          avg_fare|
+--------------+------------------+
|           EWR| 76.24024663677126|
|           N/A|  59.5731593830335|
| Staten Island|45.289861495844896|
|        Queens| 35.14462651722029|
|         Bronx| 26.26890543682963|
|      Brooklyn|18.649132800172286|
|       Unknown|14.944423051653523|
|     Manhattan|10.792468572351568|
+--------------+------------------+



In [31]:
# highest/lowest faire amounts for a trip, what burough is associated with the each
# Highest fare
df_trips.orderBy(col("fare_amount").desc()).select(
    "trip_id", "fare_amount", "pickup_borough", "dropoff_borough"
).limit(1).show()

# Lowest fare
df_trips.orderBy(col("fare_amount").asc()).select(
    "trip_id", "fare_amount", "pickup_borough", "dropoff_borough"
).limit(1).show()

+-----------+-----------+--------------+---------------+
|    trip_id|fare_amount|pickup_borough|dropoff_borough|
+-----------+-----------+--------------+---------------+
|42952172615|  623259.86|     Manhattan|      Manhattan|
+-----------+-----------+--------------+---------------+

+-----------+-----------+--------------+---------------+
|    trip_id|fare_amount|pickup_borough|dropoff_borough|
+-----------+-----------+--------------+---------------+
|42954563609|     -362.0|        Queens|         Queens|
+-----------+-----------+--------------+---------------+



In [33]:
# load the dataset from the most recently available january, is there a change to any of the average metrics.

# set dl url for most recent January trip data
# NYC TLC data is typically published with a ~2 month lag, so adjust year as needed
download_url_recent = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'

response = requests.get(download_url_recent)

recent_jan_file = "yellow_tripdata_2025-01.parquet"
if response.status_code == 200:
    with open(recent_jan_file, "wb") as f:
        f.write(response.content)

df_trips_recent = spark.read.parquet(recent_jan_file)

# apply the same derived columns as before
df_trips_recent = df_trips_recent.withColumn(
    "trip_duration_sec",
    unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")
)
df_trips_recent = df_trips_recent.withColumn("pickup_date", to_date("tpep_pickup_datetime"))
df_trips_recent = df_trips_recent.withColumn("pickup_hour", hour("tpep_pickup_datetime"))
df_trips_recent = df_trips_recent.withColumn("pickup_dow", dayofweek("tpep_pickup_datetime"))

# join to zones
df_trips_recent = df_trips_recent.join(
    df_zones.select(col("LocationID").alias("PULocationID"), col("Borough").alias("pickup_borough")),
    on="PULocationID", how="left"
)

# compare average distance/fare, 2019 vs recent
df_trips.agg(avg("trip_distance"), avg("fare_amount")).show()
df_trips_recent.agg(avg("trip_distance"), avg("fare_amount")).show()

+------------------+-----------------+
|avg(trip_distance)| avg(fare_amount)|
+------------------+-----------------+
|2.8301461681153532|12.52967677747685|
+------------------+-----------------+

+------------------+------------------+
|avg(trip_distance)|  avg(fare_amount)|
+------------------+------------------+
| 6.455646860885151|20.804253893199448|
+------------------+------------------+



### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [36]:
df_trips.createOrReplaceTempView("trips")
df_zones.createOrReplaceTempView("zones")

In [37]:
# Which trip has the highest passanger count
spark.sql("""
    SELECT *
    FROM trips
    ORDER BY passenger_count DESC
    LIMIT 1
""").show()

+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+-----------------+-----------+-----------+----------+-----------+--------------+---------------+
|DOLocationID|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|trip_duration_sec|pickup_date|pickup_hour|pickup_dow|time_bucket|pickup_borough|dropoff_borough|
+------------+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+-----

In [38]:
# which borough had most pickups? dropoffs?
spark.sql("""
    SELECT z.Borough AS pickup_borough, COUNT(*) AS pickup_count
    FROM trips t
    JOIN zones z
        ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY pickup_count DESC
""").show()

+--------------+------------+
|pickup_borough|pickup_count|
+--------------+------------+
|     Manhattan|     6950965|
|        Queens|      471173|
|       Unknown|      159815|
|      Brooklyn|       91905|
|         Bronx|       18062|
|           N/A|        3890|
|           EWR|         446|
| Staten Island|         361|
+--------------+------------+



In [39]:
# what is the average trip fare by borough?
spark.sql("""
    SELECT z.Borough AS pickup_borough, AVG(t.fare_amount) AS avg_fare
    FROM trips t
    JOIN zones z
        ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY avg_fare DESC
""").show()

+--------------+------------------+
|pickup_borough|          avg_fare|
+--------------+------------------+
|           EWR| 76.24024663677126|
|           N/A|  59.5731593830335|
| Staten Island|45.289861495844896|
|        Queens| 35.14462651722029|
|         Bronx| 26.26890543682963|
|      Brooklyn|18.649132800172286|
|       Unknown|14.944423051653523|
|     Manhattan|10.792468572351568|
+--------------+------------------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing